## Visual CNN Recommender

In [1]:
import os
import requests
import numpy as np
import pandas as pd
import joblib
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.metrics.pairwise import cosine_similarity

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/visual"
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Ready!")

Device: cpu
Ready!


In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/visual_posters.csv")
df = df.dropna(subset=["poster_path"]).reset_index(drop=True)

print(f"Movies with posters: {len(df)}")
df.head(3)

Movies with posters: 9988


,movie_id,title,poster_path
0,983044,The Arctic Convoy,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...
1,5,Four Rooms,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...
2,11,Star Wars,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...


In [3]:
print("Loading ResNet18 (pretrained)...")

# ResNet18 أخف من VGG16 وأسرع على اللاب توب
model_cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# شيل الـ classification layer الأخيرة — محتاجين الـ features بس
model_cnn = nn.Sequential(*list(model_cnn.children())[:-1])
model_cnn = model_cnn.to(device)
model_cnn.eval()

# الـ transforms المطلوبة لـ ResNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

print("ResNet18 ready!")
print(f"Feature vector size: 512")

Loading ResNet18 (pretrained)...


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Saeed/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:17<00:00, 2.74MB/s]


ResNet18 ready!
Feature vector size: 512


In [4]:
def load_image(url: str, timeout: int = 5) -> Image.Image | None:
    """تحميل صورة من URL"""
    try:
        response = requests.get(url, timeout=timeout)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content)).convert("RGB")
            return img
    except:
        pass
    return None

def extract_features(img: Image.Image) -> np.ndarray:
    """استخراج الـ features من صورة"""
    tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = model_cnn(tensor)
    return features.squeeze().cpu().numpy()

# اختبار سريع
test_url = df.iloc[0]["poster_path"]
test_img = load_image(test_url)
if test_img:
    test_feat = extract_features(test_img)
    print(f"Test OK! Feature shape: {test_feat.shape}")
    test_img.resize((150, 225))

Test OK! Feature shape: (512,)


In [5]:
# لو عندك الـ features محفوظة من قبل، حملها
features_path = f"{MODELS_DIR}/poster_features.npy"
valid_idx_path = f"{MODELS_DIR}/valid_indices.npy"

if os.path.exists(features_path):
    print("Loading saved features...")
    all_features  = np.load(features_path)
    valid_indices = np.load(valid_idx_path).tolist()
    print(f"Loaded: {len(valid_indices)} movies")

else:
    print(f"Extracting features for {len(df)} movies...")
    print("(هياخد 15-30 دقيقة حسب السرعة — بيحمّل الصور من الإنترنت)")
    
    all_features  = []
    valid_indices = []
    failed        = 0
    
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
        img = load_image(row["poster_path"])
        
        if img is None:
            failed += 1
            continue
        
        try:
            feat = extract_features(img)
            all_features.append(feat)
            valid_indices.append(i)
        except Exception as e:
            failed += 1
        
        # حفظ كل 500
        if len(valid_indices) % 500 == 0 and len(valid_indices) > 0:
            np.save(features_path, np.array(all_features))
            np.save(valid_idx_path, np.array(valid_indices))
    
    all_features  = np.array(all_features)
    valid_indices = list(valid_indices)
    
    # حفظ نهائي
    np.save(features_path, all_features)
    np.save(valid_idx_path, np.array(valid_indices))
    
    print(f"\nDone!")
    print(f"  Extracted : {len(valid_indices)} movies")
    print(f"  Failed    : {failed} movies")
    print(f"  Shape     : {all_features.shape}")

Extracting features for 9988 movies...
(هياخد 15-30 دقيقة حسب السرعة — بيحمّل الصور من الإنترنت)


Extracting:   0%|          | 0/9988 [00:00<?, ?it/s]


Done!
  Extracted : 9985 movies
  Failed    : 3 movies
  Shape     : (9985, 512)


In [6]:
print("Computing visual similarity...")

# الـ DataFrame بس للأفلام اللي عندها features
df_visual = df.iloc[valid_indices].reset_index(drop=True)

# Normalize الـ features (أسرع وأدق)
norms = np.linalg.norm(all_features, axis=1, keepdims=True)
features_norm = all_features / (norms + 1e-8)

# Cosine similarity
visual_sim = cosine_similarity(features_norm)

print(f"Visual similarity matrix: {visual_sim.shape}")
print(f"Memory: {visual_sim.nbytes / 1024 / 1024:.1f} MB")

Computing visual similarity...
Visual similarity matrix: (9985, 9985)
Memory: 380.3 MB


In [7]:
visual_indices = pd.Series(
    df_visual.index,
    index=df_visual["title"].str.lower()
).drop_duplicates()

def get_visual_recommendations(title: str, n: int = 10) -> pd.DataFrame:
    """ترشيح بناءً على تشابه البوسترات"""
    
    title_lower = title.lower().strip()
    
    if title_lower not in visual_indices:
        matches = [t for t in visual_indices.index if title_lower in t]
        if not matches:
            print(f"'{title}' not found!")
            return pd.DataFrame()
        title_lower = matches[0]
        print(f"Using: '{title_lower}'")
    
    idx = visual_indices[title_lower]
    
    sim_scores    = sorted(enumerate(visual_sim[idx]), key=lambda x: x[1], reverse=True)[1:n+1]
    movie_indices = [i[0] for i in sim_scores]
    scores        = [round(i[1], 4) for i in sim_scores]
    
    result = df_visual.iloc[movie_indices][["movie_id","title","poster_path"]].copy()
    result["visual_score"] = scores
    
    return result.reset_index(drop=True)

print("get_visual_recommendations() ready!")

get_visual_recommendations() ready!


In [8]:
from IPython.display import display, HTML

def show_recommendations_with_posters(title: str, n: int = 8):
    """عرض الترشيحات مع الصور"""
    recs = get_visual_recommendations(title, n=n)
    if recs.empty:
        return
    
    html = f"<h3>Visual recommendations for: {title}</h3>"
    html += '<div style="display:flex; flex-wrap:wrap; gap:10px;">'
    
    for _, row in recs.iterrows():
        html += f"""
        <div style="text-align:center; width:120px;">
            <img src="{row['poster_path']}" width="120" 
                 onerror="this.src='https://via.placeholder.com/120x180'">
            <p style="font-size:11px;">{row['title'][:20]}</p>
            <p style="font-size:10px; color:gray;">{row['visual_score']:.3f}</p>
        </div>"""
    
    html += "</div>"
    display(HTML(html))

# اختبار
show_recommendations_with_posters("The Dark Knight")

In [9]:
test_movies = ["The Dark Knight", "Toy Story", "The Godfather", "Inception"]

print("=== Visual Recommendations ===\n")
for movie in test_movies:
    recs = get_visual_recommendations(movie, n=5)
    if not recs.empty:
        print(f"[{movie}]")
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} visual_score: {row['visual_score']:.4f}")
        print()

=== Visual Recommendations ===

[The Dark Knight]
  → Freaks                                   visual_score: 0.8305
  → Wish                                     visual_score: 0.8272
  → One Shot                                 visual_score: 0.8163
  → Death Whisperer 2                        visual_score: 0.8138
  → Twister                                  visual_score: 0.8089

[Toy Story]
  → Toy Story 2                              visual_score: 0.8661
  → Love, Chunibyo & Other Delusions! Take On Me visual_score: 0.8566
  → Happily N'Ever After                     visual_score: 0.8556
  → Valiant                                  visual_score: 0.8534
  → Monsters University                      visual_score: 0.8479

[The Godfather]
  → The Substitute                           visual_score: 0.8372
  → Frost/Nixon                              visual_score: 0.8321
  → Misconduct                               visual_score: 0.8269
  → Jackie Brown                             visual_score:

In [10]:
print("Saving model...")

np.save(f"{MODELS_DIR}/visual_sim.npy",       visual_sim)
np.save(f"{MODELS_DIR}/features_norm.npy",    features_norm)
visual_indices.to_pickle(f"{MODELS_DIR}/visual_indices.pkl")
df_visual.to_csv(f"{MODELS_DIR}/visual_df.csv", index=False)

print("Saved:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024 / 1024
    print(f"  {f:35s} {size:.1f} MB")

Saving model...
Saved:
  features_norm.npy                   19.5 MB
  poster_features.npy                 19.5 MB
  valid_indices.npy                   0.0 MB
  visual_df.csv                       0.8 MB
  visual_indices.pkl                  0.3 MB
  visual_sim.npy                      380.3 MB
